# Error analysis (B5)

10 false positives (correct numbers flagged), 10 false negatives (fabrications missed), and where `wrong_calc` numbers fall on the detector's score (README §8, B5). The examples and tables come from `analysis/errors.py`; the explanations are ours, in our own words (README §3).

In [ ]:
import sys
from pathlib import Path

# Find the repo root, whether the notebook runs from the root or from analysis/.
ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("run this notebook inside the repo")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Image, Markdown, display

from common.io import load_config, run_dir
from analysis.errors import examples_markdown, run_all

# Another model: point CONFIG at its file, e.g. ROOT / "configs" / "llama1b.yaml" (MANUAL_STEPS.md,
# "Other models"); the folders below then become gen/<run_name>/, labels/<run_name>/, ...
CONFIG = ROOT / "config.yaml"
cfg = load_config(CONFIG)
DIR = {f: run_dir(cfg, ROOT, f) for f in ["gen", "labels", "baseline", "features", "detector", "analysis/figures"]}

DETECTOR = None    # None = best val AUROC; or "lookback", "hf", "lookback_hf", "confidence"
PICK = "random"    # or "extreme": the most confident errors

In [ ]:
detector, examples, summary = run_all(cfg, ROOT / "data" / "finqa_subset.jsonl", DIR["gen"],
                                      DIR["labels"], DIR["detector"], DIR["analysis/figures"],
                                      detector=DETECTOR, pick=PICK)
threshold = examples[0]["threshold"] if examples else None
print(f"detector: {detector}, threshold: {threshold}")

## Where `wrong_calc` numbers fall

In [ ]:
display(Image(str(DIR["analysis/figures"] / "b5_scores_by_label.png")))
pd.DataFrame(summary).set_index("label").round(3)

### Our reading (`wrong_calc`)

_In our own words._

## False positives: correct numbers flagged

In [ ]:
display(Markdown(examples_markdown([e for e in examples if e["kind"] == "fp"], detector, threshold)))

### Our reading (false positives)

_For each example, what we think explains it; then the pattern across them._

## False negatives: fabrications missed

In [ ]:
display(Markdown(examples_markdown([e for e in examples if e["kind"] == "fn"], detector, threshold)))

### Our reading (false negatives)

_For each example, what we think explains it; then the pattern across them._